In [1]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Paths
zip_path = '/content/drive/MyDrive/dataset.zip'
extract_path = '/content/dataset'

# Unzip the dataset to local Colab storage
if not os.path.exists(extract_path):
    print("Extracting dataset...")
    !unzip -q "{zip_path}" -d "{extract_path}"
    print("Extraction complete!")
else:
    print("Dataset already extracted.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset already extracted.


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Pipeline Configuration
DATASET_DIR = '/content/dataset/dataset'
IMG_SIZE = (224, 224)
BATCH_SIZE = 64  # Takes advantage of the T4 GPU's 16GB VRAM
EPOCHS = 10

# 2. Ingest & Resize Dataset
print("Loading dataset splits...")
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print(f"Mapped Classes: {class_names}")

# Optimize data loading for GPU throughput
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 3. Data Augmentation (Prevents overfitting)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# 4. MobileNetV3 Architecture for Binary Classification
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # Freeze ImageNet feature layers

inputs = layers.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)

# Apply correct scaling for MobileNetV3 [-1, 1]
x = tf.keras.applications.mobilenet_v3.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

# Single output node for binary classification (0 or 1)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs)

# 5. Compile & Train
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\n--- Training on T4 GPU ---")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Loading dataset splits...
Found 11566 files belonging to 2 classes.
Using 9253 files for training.
Found 11566 files belonging to 2 classes.
Using 2313 files for validation.
Mapped Classes: ['invalid_junk', 'vertex_valid']

--- Training on T4 GPU ---
Epoch 1/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 35s 181ms/step - accuracy: 0.8615 - loss: 0.3144 - val_accuracy: 0.9676 - val_loss: 0.1060
Epoch 2/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 7s 50ms/step - accuracy: 0.9835 - loss: 0.0822 - val_accuracy: 0.9888 - val_loss: 0.0533
Epoch 3/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 7s 49ms/step - accuracy: 0.9870 - loss: 0.0545 - val_accuracy: 0.9935 - val_loss: 0.0346
Epoch 4/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 7s 51ms/step - accuracy: 0.9912 - loss: 0.0405 - val_accuracy: 0.9935 - val_loss: 0.0264
Epoch 5/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 7s 50ms/step - accuracy: 0.9932 - loss: 0.0320 - val_accuracy: 0.9939 - val_loss: 0.0195
Epoch 6/10
145/145 ━━━━━━━━━━━━━━━━━━━━ 7s 51ms/step - accuracy: 0.9939 - loss: 0.0270 - val_accuracy:

In [3]:
from google.colab import files

print("Quantizing model to TensorFlow Lite...")

# Initialize converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Dynamic range quantization

# Convert and save
tflite_model = converter.convert()
output_filename = "vertex_classifier_quant.tflite"

with open(output_filename, "wb") as f:
    f.write(tflite_model)

print(f"Model successfully saved as {output_filename}")

# Save a backup to your Google Drive and auto-download
!cp "{output_filename}" "/content/drive/MyDrive/"
files.download(output_filename)

Quantizing model to TensorFlow Lite...
Saved artifact at '/tmp/tmpqqoffn9a'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_175')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140015633614416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140013141939728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140013141937808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140013141940688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140013141938000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140013141940496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140013141941072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140013141940880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140013141941264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140013141940304: TensorSpec(shape=(), dty

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>